## Test 04: simulated future paths with replacement and rolling WFO

This is a **conditional simulation stress test**, separate from the historical WFO. Joint equity/bond overnight and intraday returns are resampled in random-length blocks **with replacement** from a frozen historical calibration pool. Prices are compounded continuously from the last real close.

A simulation year is **252 trading sessions**, with 12 synthetic months of 21 sessions. Training uses the preceding 2,520 sessions and investment uses the next 252 sessions. These session-based windows approximate 10-year/1-year WFO; they are not actual future calendar dates. Early training mixes real and previously observed simulated data; eventually it becomes entirely simulated.

No simulated path is selected for good performance. Reported percentiles describe outcomes conditional on the calibration data and resampling assumptions, not additional independent historical evidence. Blocks retain observed joint behavior internally but break dependence at joins and cannot create crises absent from their source. See [Politis and Romano, stationary bootstrap](https://doi.org/10.1080/01621459.1994.10476870).

Requires pandas, NumPy, Matplotlib and Numba. Refresh the ETF CSV with the data notebook before running if needed.

The sampling pool defaults to the **entire joint dataset** (`CALIBRATION_YEARS = None`). This is separate from the rolling 10-year strategy training window. Per-path Sharpe tests describe simulated outcomes conditional on this generator; paths are never pooled into one historical sample.

**Default run: RSI on the USD/UBT spread.** The first code cell sets `RULE_SET = "rsi"`; change it to `"bollinger"` to run Bollinger bands instead. Only the selected rule set is used. Generated results are saved in separate `rsi` and `bollinger` directories; existing Bollinger results are from earlier runs. Use Restart Kernel and Run All after changing the selection.

In [ ]:
PAIR = ("USD", "UBT")
SIGNAL_PAIR = PAIR
RULE_SET = "rsi"
CSV_PATH = "data/etf_daily_joint.csv"
N_PATHS = 25
FUTURE_YEARS = 20
CALIBRATION_YEARS = None
TRAIN_YEARS = 10
SESSIONS_PER_YEAR = 252
MEAN_BLOCK_LENGTH = 60
MASTER_SEED = 20261028
INITIAL_CAPITAL = 100_000.0
COMMISSION_BPS = 1.0
SLIPPAGE_BPS = 4.0
MIN_TRAIN_ENTRIES = 1
SAVE_SYNTHETIC_PRICES = True

GRIDS = {'bollinger': {'length': [5, 10, 15, 20, 30, 40, 60],
               'ma_type': ['SMA', 'EMA', 'WMA'],
               'stdev': [1.0, 1.5, 2.0, 2.5, 3.0]},
 'rsi': {'rsi_window': [2, 3, 4, 5, 7, 10, 14],
         'entry_low': [5, 10, 15, 20, 25, 30],
         'entry_high': [70, 75, 80, 85, 90, 95],
         'exit_risk': [50, 60, 70, 80],
         'exit_bond': [20, 30, 40, 50]}}
COST_PATHS_BPS = {
    "Zero cost": (0.0, 0.0),
    "Base cost": (COMMISSION_BPS, SLIPPAGE_BPS),
    "10 bps per leg": (1.0, 9.0),
    "20 bps per leg": (1.0, 19.0),
}

SHARPE_NULL = 0.5
SHARPE_ALPHA = 0.05
SHARPE_BOOTSTRAPS = 10_000
SHARPE_PRIMARY_BLOCK = 20
SHARPE_BLOCK_LENGTHS = [5, 10, 20, 60]
SHARPE_HAC_LAGS = 20
SHARPE_TEST_SEED = 20261028
SAVE_SHARPE_DRAWS = False


## Freeze calibration data and reproduce the existing indicator/rule definitions

In [ ]:
from pathlib import Path
from itertools import product
from datetime import datetime, timezone
import hashlib
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display
from numba import njit

risk, bond = PAIR
assert risk != bond and len(SIGNAL_PAIR) == 2
assert N_PATHS > 0 and FUTURE_YEARS > 0 and TRAIN_YEARS > 0
assert MEAN_BLOCK_LENGTH >= 1 and SESSIONS_PER_YEAR % 12 == 0
source_path = Path(CSV_PATH)
prices = pd.read_csv(source_path, index_col="date", parse_dates=["date"]).sort_index()
assert prices.index.is_unique and prices.index.is_monotonic_increasing
TICKERS = list(dict.fromkeys([*PAIR, *SIGNAL_PAIR]))
required = [f"{t}_{field}" for t in TICKERS for field in ["open", "close", "adj_close"]]
if not set(required).issubset(prices.columns):
    raise ValueError("Missing selected tickers; run the data notebook")
if not np.isfinite(prices[required].to_numpy()).all() or (prices[required] <= 0).any().any():
    raise ValueError("Prices must be complete, finite and positive")
n_history = len(prices)
train_sessions = TRAIN_YEARS * SESSIONS_PER_YEAR
future_sessions = FUTURE_YEARS * SESSIONS_PER_YEAR
if n_history < train_sessions:
    raise ValueError("Insufficient real history for the initial training window")
origin = prices.index[-1]
calibration_start = prices.index[0] if CALIBRATION_YEARS is None else origin - pd.DateOffset(years=CALIBRATION_YEARS)
historical_close = np.column_stack([prices[f"{t}_adj_close"] for t in TICKERS])
historical_open = np.column_stack([
    prices[f"{t}_open"] * prices[f"{t}_adj_close"] / prices[f"{t}_close"] for t in TICKERS
])
source_overnight = np.zeros_like(historical_close)
source_overnight[1:] = historical_open[1:] / historical_close[:-1] - 1
source_intraday = historical_close / historical_open - 1
pool_rows = np.flatnonzero((prices.index >= calibration_start) & (np.arange(n_history) > 0))
if len(pool_rows) < MEAN_BLOCK_LENGTH:
    raise ValueError("Calibration pool is too short")
cost_names = list(COST_PATHS_BPS)
assert "Base cost" in cost_names and "Zero cost" in cost_names
base_cost_id = cost_names.index("Base cost")
zero_cost_id = cost_names.index("Zero cost")
cost_values = np.array(list(COST_PATHS_BPS.values()), dtype=float) / 10_000
assert np.isfinite(cost_values).all() and (cost_values >= 0).all() and (cost_values < 1).all()
np.testing.assert_allclose(cost_values[base_cost_id], [COMMISSION_BPS / 10000, SLIPPAGE_BPS / 10000])
np.testing.assert_allclose(cost_values[zero_cost_id], [0, 0])
calibration_tag = "calibration_all" if CALIBRATION_YEARS is None else f"calibration_{CALIBRATION_YEARS}y"
output_dir = Path("data") / f"test_04_simulations_{risk}_{bond}" / RULE_SET / calibration_tag / f"seed_{MASTER_SEED}_block_{MEAN_BLOCK_LENGTH}"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Frozen calibration: {prices.index[pool_rows[0]].date()} to {origin.date()}, {len(pool_rows):,} sessions")
print(f"{N_PATHS} paths x {FUTURE_YEARS} synthetic years = {N_PATHS * FUTURE_YEARS} simulated investment windows")


In [ ]:
def wilder_rsi(series, window):
    window = int(window)
    if window < 1 or window >= len(series):
        raise ValueError("RSI window must be positive and shorter than the sample")
    change = series.diff()
    gains = change.clip(lower=0)
    losses = -change.clip(upper=0)
    avg_gain = gains.iloc[1:window + 1].mean()
    avg_loss = losses.iloc[1:window + 1].mean()
    values = pd.Series(np.nan, index=series.index)
    for i in range(window, len(series)):
        if i > window:
            avg_gain = ((window - 1) * avg_gain + gains.iloc[i]) / window
            avg_loss = ((window - 1) * avg_loss + losses.iloc[i]) / window
        values.iloc[i] = (50 if avg_gain == 0 and avg_loss == 0 else
                          100 if avg_loss == 0 else 100 - 100 / (1 + avg_gain / avg_loss))
    return values

def moving_average(series, length, ma_type):
    if ma_type == "SMA":
        return series.rolling(length, min_periods=length).mean()
    if ma_type == "EMA":
        return series.ewm(span=length, adjust=False, min_periods=length).mean()
    if ma_type == "WMA":
        weights = np.arange(1, length + 1, dtype=float)
        return series.rolling(length, min_periods=length).apply(
            lambda values: np.dot(values, weights) / weights.sum(), raw=True
        )
    raise ValueError(f"Unsupported MA type: {ma_type}")

def valid_parameters(parameters):
    if RULE_SET == "bollinger":
        return (
            parameters.length.ge(2) & parameters.length.lt(len(prices))
            & parameters.length.eq(parameters.length.astype(int))
            & parameters.ma_type.isin(["SMA", "EMA", "WMA"])
            & parameters.stdev.gt(0) & np.isfinite(parameters.stdev)
        )
    if RULE_SET == "rsi":
        return (
            parameters.rsi_window.ge(1) & parameters.rsi_window.lt(len(prices))
            & parameters.rsi_window.eq(parameters.rsi_window.astype(int))
            & parameters.entry_low.ge(0) & parameters.entry_high.le(100)
            & parameters.entry_low.lt(parameters.exit_bond)
            & parameters.exit_bond.le(parameters.exit_risk)
            & parameters.exit_risk.lt(parameters.entry_high)
        )
    raise ValueError("RULE_SET must be bollinger or rsi")

def build_features(data, parameters):
    frame = pd.DataFrame(index=data.index)
    frame["spread"] = data[f"{SIGNAL_PAIR[0]}_adj_close"] / data[f"{SIGNAL_PAIR[1]}_adj_close"]
    if RULE_SET == "bollinger":
        for length in sorted(parameters.length.unique()):
            frame[f"bb_std_{int(length)}"] = frame.spread.rolling(int(length), min_periods=int(length)).std(ddof=0)
        for length, ma_type in parameters[["length", "ma_type"]].drop_duplicates().itertuples(index=False, name=None):
            frame[f"bb_middle_{int(length)}_{ma_type}"] = moving_average(frame.spread, int(length), ma_type)
    else:
        for window in sorted(parameters.rsi_window.unique()):
            frame[f"rsi_{int(window)}"] = wilder_rsi(frame.spread, int(window))
    return frame

def make_rules(features, parameters):
    if RULE_SET == "bollinger":
        spread_values = features.spread.to_numpy()[:, None]
        middle = features[[f"bb_middle_{int(row.length)}_{row.ma_type}" for row in parameters.itertuples()]].to_numpy()
        width = features[[f"bb_std_{int(length)}" for length in parameters.length]].to_numpy() * parameters.stdev.to_numpy()
        return {
            "enter_risk": spread_values < middle - width,
            "enter_bond": spread_values > middle + width,
            "exit_risk": spread_values > middle,
            "exit_bond": spread_values < middle,
        }
    values = features[[f"rsi_{int(window)}" for window in parameters.rsi_window]].to_numpy()
    return {
        "enter_risk": values < parameters.entry_low.to_numpy(),
        "enter_bond": values > parameters.entry_high.to_numpy(),
        "exit_risk": values > parameters.exit_risk.to_numpy(),
        "exit_bond": values < parameters.exit_bond.to_numpy(),
    }

parameter_names = list(GRIDS[RULE_SET])
parameters = pd.DataFrame(product(*GRIDS[RULE_SET].values()), columns=parameter_names)
parameters = parameters.loc[valid_parameters(parameters)].sort_values(parameter_names).reset_index(drop=True)
if parameters.empty:
    raise ValueError("Empty parameter grid")
print(f"{len(parameters)} training candidates per investment window")


## Joint stationary bootstrap: random-length blocks with replacement

All required tickers use the same source row on each simulated session. Source blocks wrap within the frozen calibration pool. Only returns are stitched; historical price levels are never pasted together.

In [ ]:
def sample_blocks(seed):
    rng = np.random.default_rng(seed)
    sampled = np.empty(future_sessions, dtype=np.int64)
    block_ids = np.empty(future_sessions, dtype=np.int64)
    block_id = -1
    cursor = 0
    for i in range(future_sessions):
        restart = i == 0 or rng.random() < 1 / MEAN_BLOCK_LENGTH
        if restart:
            cursor = int(rng.integers(len(pool_rows)))
            block_id += 1
        else:
            cursor = (cursor + 1) % len(pool_rows)
        sampled[i] = pool_rows[cursor]
        block_ids[i] = block_id
    return sampled, block_ids

def reconstruct_path(sampled):
    opening = np.empty((n_history + future_sessions, len(TICKERS)))
    closing = np.empty_like(opening)
    opening[:n_history] = historical_open
    closing[:n_history] = historical_close
    for i, source_row in enumerate(sampled, start=n_history):
        opening[i] = closing[i - 1] * (1 + source_overnight[source_row])
        closing[i] = opening[i] * (1 + source_intraday[source_row])
    frame = pd.DataFrame(index=pd.RangeIndex(len(closing), name="session"))
    for j, ticker in enumerate(TICKERS):
        frame[f"{ticker}_open"] = opening[:, j]
        frame[f"{ticker}_close"] = closing[:, j]
        frame[f"{ticker}_adj_close"] = closing[:, j]
    np.testing.assert_allclose(opening[n_history:] / closing[n_history - 1:-1] - 1,
                               source_overnight[sampled], rtol=1e-10, atol=1e-12)
    np.testing.assert_allclose(closing[n_history:] / opening[n_history:] - 1,
                               source_intraday[sampled], rtol=1e-10, atol=1e-12)
    return frame

def engine_inputs(frame):
    f = build_features(frame, parameters)
    masks = make_rules(f, parameters)
    rules = np.stack([masks[k] for k in ["enter_risk", "enter_bond", "exit_risk", "exit_bond"]])
    if np.any(rules[0] & rules[1]):
        raise ValueError("Conflicting entries")
    close = np.column_stack([frame[f"{t}_adj_close"] for t in PAIR])
    op = np.column_stack([frame[f"{t}_open"] for t in PAIR])
    over = np.zeros((len(frame), 3))
    intra = np.zeros_like(over)
    over[1:, 1:] = op[1:] / close[:-1] - 1
    intra[:, 1:] = close / op - 1
    return f, np.ascontiguousarray(rules), over, intra


## Compiled WFO engine: training-only net Sharpe, frozen annual parameters, next-open execution

Each candidate starts training in cash. The actual simulated portfolio carries holdings across investment blocks. Costs follow the same executed-notional convention as the historical notebook. Every cost scenario uses the base-cost-selected parameters and identical holding decisions.

In [ ]:
@njit
def advance_state(state, rules, execution_session, config_id):
    if config_id < 0:
        return 0
    signal = execution_session - 1
    if signal < 0:
        return state
    if (state == 1 and rules[2, signal, config_id]) or (state == 2 and rules[3, signal, config_id]):
        state = 0
    if state == 0:
        if rules[0, signal, config_id]:
            state = 1
        elif rules[1, signal, config_id]:
            state = 2
    return state

@njit
def cost_factor(old, new, commission, slippage):
    sell = old != new and old != 0
    buy = old != new and new != 0
    after_sale = (1 - slippage) * (1 - commission) if sell else 1.0
    factor = after_sale / ((1 + slippage) * (1 + commission)) if buy else after_sale
    paid_commission = (1 - slippage) * commission if sell else 0.0
    paid_slippage = slippage if sell else 0.0
    if buy:
        paid_commission += after_sale * commission / (1 + commission)
        paid_slippage += after_sale * slippage / ((1 + commission) * (1 + slippage))
    return factor, paid_commission, paid_slippage, int(sell) + int(buy), int(buy)

@njit
def score_training(over, intra, rules, start, end, commission, slippage, min_entries):
    n = rules.shape[2]
    scores = np.full(n, np.nan)
    total_return = np.zeros(n)
    max_drawdown = np.zeros(n)
    entries = np.zeros(n, dtype=np.int64)
    for config in range(n):
        state = 0
        wealth = 1.0
        peak = 1.0
        total = 0.0
        squared = 0.0
        worst = 0.0
        count_entries = 0
        for day in range(start, end):
            old = state
            state = advance_state(state, rules, day, config)
            factor, _, _, _, buy = cost_factor(old, state, commission, slippage)
            r = (1 + over[day, old]) * factor * (1 + intra[day, state]) - 1
            total += r
            squared += r * r
            wealth *= 1 + r
            peak = max(peak, wealth)
            worst = min(worst, wealth / peak - 1)
            count_entries += buy
        count = end - start
        mean = total / count
        std = np.sqrt(max((squared - count * mean * mean) / (count - 1), 0.0))
        if std > 1e-14 and count_entries >= min_entries:
            scores[config] = mean / std * np.sqrt(252.0)
        total_return[config] = wealth - 1
        max_drawdown[config] = worst
        entries[config] = count_entries
    return scores, total_return, max_drawdown, entries

@njit
def run_simulated_wfo(over, intra, rules, history_count, training_length, years, year_length,
                      capital, commission, slippage, min_entries, scenarios, base_id):
    n_future = years * year_length
    equity = np.full((scenarios.shape[0], n_future), capital)
    wealth = np.full(scenarios.shape[0], capital)
    daily = np.zeros((n_future, 11))
    folds = np.zeros((years, 8))
    grid_scores = np.empty((years, rules.shape[2]))
    state = 0
    gross_wealth = capital
    for year in range(years):
        end = history_count + year * year_length
        start = end - training_length
        scores, train_returns, train_dd, train_entries = score_training(
            over, intra, rules, start, end, commission, slippage, min_entries
        )
        grid_scores[year] = scores
        winner = -1
        best_score = -np.inf
        for j in range(len(scores)):
            if np.isfinite(scores[j]) and scores[j] > best_score:
                winner = j
                best_score = scores[j]
        folds[year, 0] = winner
        folds[year, 1] = best_score if winner >= 0 else np.nan
        folds[year, 2] = train_returns[winner] if winner >= 0 else 0.0
        folds[year, 3] = train_dd[winner] if winner >= 0 else 0.0
        folds[year, 4] = train_entries[winner] if winner >= 0 else 0
        beginning = wealth[base_id]
        sum_r = 0.0
        sum_r2 = 0.0
        for day in range(end, end + year_length):
            row = day - history_count
            old = state
            state = advance_state(state, rules, day, winner)
            open_factor = 1 + over[day, old]
            intra_factor = 1 + intra[day, state]
            beginning_day = wealth[base_id]
            base_factor, comm_rate, slip_rate, legs, buy = cost_factor(old, state, commission, slippage)
            after_cost_open = beginning_day * open_factor * base_factor
            equity_pnl = (beginning_day * over[day, old] if old == 1 else 0.0) + (after_cost_open * intra[day, state] if state == 1 else 0.0)
            bond_pnl = (beginning_day * over[day, old] if old == 2 else 0.0) + (after_cost_open * intra[day, state] if state == 2 else 0.0)
            paid_comm = beginning_day * open_factor * comm_rate
            paid_slip = beginning_day * open_factor * slip_rate
            net_return = open_factor * base_factor * intra_factor - 1
            gross_return = open_factor * intra_factor - 1
            gross_wealth *= 1 + gross_return
            for scenario in range(scenarios.shape[0]):
                factor, _, _, _, _ = cost_factor(old, state, scenarios[scenario, 0], scenarios[scenario, 1])
                wealth[scenario] *= open_factor * factor * intra_factor
                equity[scenario, row] = wealth[scenario]
            daily[row, 0] = net_return
            daily[row, 1] = gross_return
            daily[row, 2] = beginning_day
            daily[row, 3] = equity_pnl
            daily[row, 4] = bond_pnl
            daily[row, 5] = paid_comm
            daily[row, 6] = paid_slip
            daily[row, 7] = legs
            daily[row, 8] = old
            daily[row, 9] = state
            daily[row, 10] = gross_wealth
            sum_r += net_return
            sum_r2 += net_return * net_return
        folds[year, 5] = wealth[base_id] / beginning - 1
        mean = sum_r / year_length
        std = np.sqrt(max((sum_r2 - year_length * mean * mean) / (year_length - 1), 0.0))
        folds[year, 6] = mean / std * np.sqrt(252.0) if std > 1e-14 else np.nan
        folds[year, 7] = wealth[base_id]
    return equity, daily, folds, grid_scores


## Run every pre-specified seed and export source provenance

Every path is retained. Future observations are labeled simulated; no future historical data enters the calibration pool. Synthetic price files contain adjusted-equivalent opens/closes, not original exchange tape prices.

In [ ]:
def assert_daily_accounting(frame, context=""):
    start = frame.starting_equity.to_numpy(dtype=float)
    end = frame.net_equity.to_numpy(dtype=float)
    expected = frame.net_return.to_numpy(dtype=float)
    terms = np.column_stack([
        frame.equity_pnl.to_numpy(dtype=float),
        frame.bond_pnl.to_numpy(dtype=float),
        -frame.commission.to_numpy(dtype=float),
        -frame.slippage.to_numpy(dtype=float),
    ])
    if (not np.isfinite(start).all() or not np.isfinite(end).all()
            or not np.isfinite(expected).all() or not np.isfinite(terms).all()
            or (start <= 0).any() or (end <= 0).any()):
        raise AssertionError(f"{context}: accounting values must be finite and equity positive")
    term_rates = terms / start[:, None]
    pnl_rate = term_rates.sum(axis=1)
    wealth_rate = end / start - 1
    tolerance = 64 * np.finfo(np.float64).eps * (
        1 + np.abs(term_rates).sum(axis=1) + np.abs(expected)
    )
    for label, observed in [("PnL attribution", pnl_rate), ("wealth change", wealth_rate)]:
        residual = np.abs(observed - expected)
        bad = residual > tolerance
        if bad.any():
            row = int(np.argmax(residual / tolerance))
            raise AssertionError(
                f"{context}: {label} mismatch at row {row}; return residual "
                f"{residual[row]:.6g} exceeds floating-point bound {tolerance[row]:.6g}"
            )
    return {
        "max_pnl_return_residual": float(np.max(np.abs(pnl_rate - expected))),
        "max_wealth_return_residual": float(np.max(np.abs(wealth_rate - expected))),
        "max_dollar_residual": float(np.max(np.abs(terms.sum(axis=1) - (end - start)))),
        "max_equity": float(end.max()),
    }

seed_sequences = np.random.SeedSequence(MASTER_SEED).spawn(N_PATHS)
seeds = [int(seed.generate_state(1)[0]) for seed in seed_sequences]
assert len(set(seeds)) == N_PATHS
all_equities = np.empty((N_PATHS, len(cost_names), future_sessions))
all_daily = []
accounting_diagnostics = []
all_folds = []
all_grids = []
all_sources = []
all_synthetic_prices = []
first_frame = None
first_inputs = None
first_kernel_result = None
started = time.perf_counter()
for path_id, seed in enumerate(seeds):
    sampled, block_ids = sample_blocks(seed)
    frame = reconstruct_path(sampled)
    f, rules, over, intra = engine_inputs(frame)
    equity, records, fold_values, scores = run_simulated_wfo(
        over, intra, rules, n_history, train_sessions, FUTURE_YEARS, SESSIONS_PER_YEAR,
        INITIAL_CAPITAL, COMMISSION_BPS / 10000, SLIPPAGE_BPS / 10000,
        MIN_TRAIN_ENTRIES, cost_values, base_cost_id
    )
    all_equities[path_id] = equity
    d = pd.DataFrame(records, columns=["net_return", "gross_return", "starting_equity", "equity_pnl", "bond_pnl",
                                       "commission", "slippage", "transaction_legs", "previous_state", "state", "gross_equity"])
    d["net_equity"] = equity[base_cost_id]
    d["config_id"] = np.repeat(fold_values[:, 0].astype(int), SESSIONS_PER_YEAR)
    d["path"] = path_id
    d["simulation_session"] = np.arange(1, future_sessions + 1)
    d["simulation_year"] = np.arange(future_sessions) // SESSIONS_PER_YEAR + 1
    d["simulation_month"] = np.arange(future_sessions) // (SESSIONS_PER_YEAR // 12) + 1
    for col in ["previous_state", "state", "transaction_legs"]:
        d[col] = d[col].astype(int)
    accounting_diagnostics.append({"path": path_id, **assert_daily_accounting(d, f"Path {path_id}")})
    np.testing.assert_allclose(equity[zero_cost_id], d.gross_equity, rtol=1e-10)
    np.testing.assert_allclose((1 + d.net_return).cumprod() * INITIAL_CAPITAL, d.net_equity, rtol=1e-9)
    all_daily.append(d)
    for year in range(FUTURE_YEARS):
        config_id = int(fold_values[year, 0])
        end = n_history + year * SESSIONS_PER_YEAR
        start = end - train_sessions
        real_count = max(min(end, n_history) - start, 0)
        details = json.loads(parameters.loc[[config_id]].to_json(orient="records"))[0] if config_id >= 0 else {k: None for k in parameter_names}
        all_folds.append({"path": path_id, "seed": seed, "simulation_year": year + 1,
                          "train_start_index": start, "train_end_exclusive": end,
                          "investment_start_index": end, "investment_end_exclusive": end + SESSIONS_PER_YEAR,
                          "train_real_sessions": real_count, "train_simulated_sessions": train_sessions - real_count,
                          "config_id": config_id, "train_net_sharpe": fold_values[year, 1],
                          "train_net_return": fold_values[year, 2], "train_max_drawdown": fold_values[year, 3],
                          "train_entries": int(fold_values[year, 4]), "oos_net_return": fold_values[year, 5],
                          "oos_net_sharpe": fold_values[year, 6], **details})
        g = parameters.copy()
        g["train_net_sharpe"] = scores[year]
        g["path"] = path_id
        g["simulation_year"] = year + 1
        g["config_id"] = np.arange(len(parameters))
        all_grids.append(g)
    all_sources.append(pd.DataFrame({"path": path_id, "simulation_session": np.arange(1, future_sessions + 1),
                                     "block_id": block_ids, "source_row": sampled,
                                     "source_date": prices.index[sampled], "source_kind": "simulated"}))
    if SAVE_SYNTHETIC_PRICES:
        future = frame.iloc[n_history:].reset_index(drop=True).copy()
        future.insert(0, "simulation_session", np.arange(1, future_sessions + 1))
        future.insert(0, "path", path_id)
        future["source_kind"] = "simulated"
        all_synthetic_prices.append(future)
    if path_id == 0:
        first_frame = frame.copy()
        first_inputs = (f, rules, over, intra)
        first_kernel_result = (equity.copy(), records.copy(), fold_values.copy(), scores.copy())
    print(f"Path {path_id + 1}/{N_PATHS} complete; seed {seed}; elapsed {time.perf_counter() - started:.1f}s", flush=True)
daily_results = pd.concat(all_daily, ignore_index=True)
folds = pd.DataFrame(all_folds)
training_results = pd.concat(all_grids, ignore_index=True)
source_trace = pd.concat(all_sources, ignore_index=True)

accounting_diagnostics = pd.DataFrame(accounting_diagnostics).set_index("path")


## Distributions across all simulated paths; historical evidence remains separate

In [ ]:
def path_metrics(returns):
    r = np.asarray(returns, dtype=float)
    growth = np.cumprod(1 + r)
    peaks = np.maximum.accumulate(np.maximum(growth, 1.0))
    volatility = r.std(ddof=1) * np.sqrt(252)
    return {"CAGR": growth[-1] ** (1 / FUTURE_YEARS) - 1,
            "Sharpe": r.mean() * 252 / volatility if volatility > 0 else np.nan,
            "Max drawdown": float((growth / peaks - 1).min()),
            "Total return": growth[-1] - 1, "Terminal wealth": growth[-1] * INITIAL_CAPITAL}

metrics = pd.DataFrame([{"path": path_id, "seed": seeds[path_id],
                         **path_metrics(daily_results.loc[daily_results.path == path_id, "net_return"])}
                        for path_id in range(N_PATHS)]).set_index("path")
summary = metrics[["CAGR", "Sharpe", "Max drawdown", "Total return", "Terminal wealth"]].quantile([.05, .25, .5, .75, .95])
summary.index = ["5th percentile", "25th percentile", "Median", "75th percentile", "95th percentile"]
display(summary.style.format({"CAGR": "{:.2%}", "Sharpe": "{:.3f}", "Max drawdown": "{:.2%}",
                              "Total return": "{:.2%}", "Terminal wealth": "{:,.0f}"}))
print(f"Losing full paths: {metrics['Total return'].lt(0).mean():.1%}")
print(f"Losing investment years: {folds.oos_net_return.lt(0).mean():.1%}")
historical_file = Path("data") / f"test_03_wfo_{risk}_{bond}" / RULE_SET / "train_120m_test_12m" / "assessment.csv"
if historical_file.exists():
    print("Historical WFO reference only — separate sample and period; not included in simulation percentiles")
    display(pd.read_csv(historical_file, index_col=0).loc[["WFO net"]])

selection_frequency = folds.groupby(parameter_names, dropna=False).size().rename("selections").reset_index()
selection_frequency["share"] = selection_frequency.selections / len(folds)
selection_frequency = selection_frequency.sort_values("selections", ascending=False).reset_index(drop=True)
print("Most frequently selected configurations across all simulated investment windows")
display(selection_frequency.head(10))


In [ ]:
base_paths = np.column_stack([np.full(N_PATHS, INITIAL_CAPITAL), all_equities[:, base_cost_id]])
x = np.arange(future_sessions + 1) / SESSIONS_PER_YEAR
q05, q25, q50, q75, q95 = np.quantile(base_paths, [.05, .25, .5, .75, .95], axis=0)
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True, constrained_layout=True)
axes[0].fill_between(x, q05, q95, alpha=.15, label="5th–95th simulation percentiles")
axes[0].fill_between(x, q25, q75, alpha=.25, label="25th–75th simulation percentiles")
axes[0].plot(x, q50, label="Pointwise median", color="black")
for path_id in range(min(5, N_PATHS)):
    axes[0].plot(x, base_paths[path_id], linewidth=.6, alpha=.3)
axes[0].set_yscale("log")
axes[0].set_ylabel("Net portfolio value (log)")
axes[0].set_title(f"{risk}/{bond} {RULE_SET}: simulated WFO; all {N_PATHS} paths retained")
axes[0].legend()
peaks = np.maximum.accumulate(base_paths, axis=1)
dd = base_paths / peaks - 1
low, median, high = np.quantile(dd, [.05, .5, .95], axis=0)
axes[1].fill_between(x, low, high, alpha=.2)
axes[1].plot(x, median, color="black", label="Pointwise median drawdown")
axes[1].set_xlabel("Simulated years (252 sessions each)")
axes[1].set_ylabel("Drawdown")
axes[1].yaxis.set_major_formatter(PercentFormatter(1))
for ax in axes: ax.grid(alpha=.25)
fig.savefig(output_dir / "simulation_equity_fan.png", dpi=140)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for column, ax in zip(["CAGR", "Sharpe", "Max drawdown"], axes):
    ax.hist(metrics[column].dropna(), bins=min(12, N_PATHS), edgecolor="white")
    ax.set_title(column)
    if column != "Sharpe": ax.xaxis.set_major_formatter(PercentFormatter(1))
fig.suptitle("Conditional simulation outcomes — no best-path selection")
fig.savefig(output_dir / "simulation_metric_distributions.png", dpi=140)
plt.show()


## Annual net-return heatmap: every path, every simulated investment year

In [ ]:
annual_returns = folds.pivot(index="path", columns="simulation_year", values="oos_net_return")
values = annual_returns.to_numpy()
limit = max(float(np.nanmax(np.abs(values))), .001)
fig, ax = plt.subplots(figsize=(18, max(7, N_PATHS * .35)), constrained_layout=True)
image = ax.imshow(values, cmap="RdYlGn", aspect="auto", vmin=-limit, vmax=limit)
ax.set_xticks(range(FUTURE_YEARS), [f"Y{year}" for year in range(1, FUTURE_YEARS + 1)])
ax.set_yticks(range(N_PATHS), [f"Path {i}" for i in range(N_PATHS)])
if N_PATHS <= 40 and FUTURE_YEARS <= 30:
    for row in range(N_PATHS):
        for col in range(FUTURE_YEARS):
            value = values[row, col]
            ax.text(col, row, f"{value:.0%}", ha="center", va="center", fontsize=7,
                    color="white" if abs(value) > .65 * limit else "black")
fig.colorbar(image, ax=ax).ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_title("SIMULATED OOS annual net returns — all paths")
fig.savefig(output_dir / "simulation_annual_heatmap.png", dpi=140)
plt.show()


## Trading-cost paths: identical parameters and holding decisions across cost assumptions

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, constrained_layout=True)
for i, name in enumerate(cost_names):
    median_path = np.median(all_equities[:, i], axis=0)
    axes[0].plot(x, np.r_[INITIAL_CAPITAL, median_path], label=name)
axes[0].set_yscale("log")
axes[0].set_ylabel("Median portfolio value")
axes[0].legend()
paid = daily_results.groupby("simulation_session")[["commission", "slippage"]].mean().cumsum()
axes[1].plot(x, np.r_[0, paid.commission], label="Mean cumulative commission paid")
axes[1].plot(x, np.r_[0, paid.slippage], label="Mean cumulative slippage paid")
axes[1].set_ylabel("Amount paid")
axes[1].set_xlabel("Simulated years")
axes[1].legend()
for ax in axes: ax.grid(alpha=.25)
fig.suptitle("Simulated WFO cost paths; training optimized under base costs")
fig.savefig(output_dir / "simulation_cost_paths.png", dpi=140)
plt.show()


## Return-attribution protocol: equities, bonds and negative trading-cost contributions

Each path-year attribution sums exactly to its net return. The chart averages contributions across paths, so its components also add up to the mean annual return.

In [ ]:
pnl = daily_results[["path", "simulation_year", "equity_pnl", "bond_pnl", "commission", "slippage"]].copy()
pnl["commission"] *= -1
pnl["slippage"] *= -1
annual_pnl = pnl.groupby(["path", "simulation_year"])[["equity_pnl", "bond_pnl", "commission", "slippage"]].sum()
starts = daily_results.groupby(["path", "simulation_year"]).starting_equity.first()
attribution = annual_pnl.div(starts, axis=0).rename(columns={"equity_pnl": "Equity", "bond_pnl": "Bonds", "commission": "Commission", "slippage": "Slippage"})
attribution["Total"] = folds.set_index(["path", "simulation_year"]).oos_net_return
np.testing.assert_allclose(attribution.drop(columns="Total").sum(axis=1), attribution.Total, rtol=1e-8, atol=1e-11)
mean_contributions = attribution.groupby("simulation_year").mean()
x_year = np.arange(FUTURE_YEARS)
positive = np.zeros(FUTURE_YEARS)
negative = np.zeros(FUTURE_YEARS)
fig, ax = plt.subplots(figsize=(15, 6), constrained_layout=True)
for name, color in [("Equity", "tab:blue"), ("Bonds", "tab:orange"), ("Commission", "tab:red"), ("Slippage", "tab:purple")]:
    value = mean_contributions[name].to_numpy()
    ax.bar(x_year, value, bottom=np.where(value >= 0, positive, negative), color=color, label=name)
    positive += np.maximum(value, 0)
    negative += np.minimum(value, 0)
ax.scatter(x_year, mean_contributions.Total, color="black", marker="D", label="Mean net return", s=18, zorder=3)
ax.set_xticks(x_year, [f"Y{i}" for i in range(1, FUTURE_YEARS + 1)])
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.set_ylabel("Mean annual contribution (percentage points)")
ax.set_title("SIMULATED WFO: average equity/bond/cost return contributions")
ax.axhline(0, color="gray", linewidth=.7)
ax.grid(axis="y", alpha=.2)
ax.legend()
fig.savefig(output_dir / "simulation_return_contributions.png", dpi=140)
plt.show()


## Transaction-leg audit trail and monthly/yearly return tables

Synthetic month 1–12 contains 21 sessions each. Signal session 0 means the last real close at the simulation origin.

In [ ]:
trade_rows = []
commission = COMMISSION_BPS / 10000
slippage = SLIPPAGE_BPS / 10000
labels = ["CASH", risk, bond]
ticker_columns = {ticker: i for i, ticker in enumerate(TICKERS)}
np.testing.assert_array_equal(source_trace.path, daily_results.path)
np.testing.assert_array_equal(source_trace.simulation_session, daily_results.simulation_session)
for row, source_row in zip(daily_results.itertuples(index=False), source_trace.source_row):
    old, new = row.previous_state, row.state
    if old == new:
        continue
    opening_wealth = row.starting_equity
    if old:
        opening_wealth *= 1 + source_overnight[source_row, ticker_columns[labels[old]]]
    remaining = opening_wealth
    if old:
        notional = remaining * (1 - slippage)
        trade_rows.append({"path": row.path, "simulation_session": row.simulation_session,
                           "signal_simulation_session": row.simulation_session - 1,
                           "side": "SELL", "asset": labels[old], "executed_notional": notional,
                           "commission": notional * commission, "slippage": remaining * slippage})
        remaining = notional * (1 - commission)
    if new:
        notional = remaining / (1 + commission)
        trade_rows.append({"path": row.path, "simulation_session": row.simulation_session,
                           "signal_simulation_session": row.simulation_session - 1,
                           "side": "BUY", "asset": labels[new], "executed_notional": notional,
                           "commission": notional * commission, "slippage": notional * slippage / (1 + slippage)})
trade_columns = ["path", "simulation_session", "signal_simulation_session", "side", "asset",
                 "executed_notional", "commission", "slippage"]
trades = pd.DataFrame(trade_rows, columns=trade_columns)
monthly = daily_results.groupby(["path", "simulation_month"]).net_return.apply(lambda r: (1 + r).prod() - 1).rename("return").reset_index()
monthly["simulation_year"] = (monthly.simulation_month - 1) // 12 + 1
monthly["month_in_year"] = (monthly.simulation_month - 1) % 12 + 1
monthly_table = monthly.pivot(index=["path", "simulation_year"], columns="month_in_year", values="return")
monthly_table["Year"] = folds.set_index(["path", "simulation_year"]).oos_net_return
np.testing.assert_allclose((1 + monthly_table.drop(columns="Year")).prod(axis=1) - 1,
                           monthly_table.Year, rtol=1e-8, atol=1e-11)


## Simulation, leakage and accounting audit

The frozen calibration pool is known at the simulation origin. Generated observations are revealed to the strategy only in simulated chronological order. This audit checks causal use of generated data; resampling historical observations into hypothetical futures is explicitly part of the simulation model.

In [ ]:
audit_rows = []
def audit(name, passed=True):
    if not bool(passed): raise AssertionError(name)
    audit_rows.append({"check": name, "status": "PASS"})

sample0, blocks0 = sample_blocks(seeds[0])
sample0_repeat, blocks0_repeat = sample_blocks(seeds[0])
audit("Fixed seed reproduces sampled source blocks", np.array_equal(sample0, sample0_repeat) and np.array_equal(blocks0, blocks0_repeat))
audit("Resampling uses only the frozen historical calibration pool", source_trace.source_row.isin(pool_rows).all() and source_trace.source_date.le(origin).all())
saved_pool = pool_rows
try:
    pool_rows = saved_pool[:1]
    reused, _ = sample_blocks(seeds[0])
    assert len(reused) > 1 and np.unique(reused).size == 1
finally:
    pool_rows = saved_pool
audit("Sampler allows source reuse with replacement")
audit("All required assets share each sampled source row", first_frame.shape[1] == 3 * len(TICKERS))
audit("Price reconstruction matches source overnight/intraday returns without pasted level jumps")
audit("Every pre-specified seed/path/year is retained", len(metrics) == N_PATHS and len(folds) == N_PATHS * FUTURE_YEARS)
audit("Training indices end before investment indices", (folds.train_end_exclusive == folds.investment_start_index).all() and ((folds.train_end_exclusive - folds.train_start_index) == train_sessions).all())
audit("Rolling training uses only real history and already-observed simulated sessions", (folds.train_real_sessions + folds.train_simulated_sessions).eq(train_sessions).all())
for path_id, group in daily_results.groupby("path", sort=False):
    assert group.previous_state.iloc[0] == 0
    np.testing.assert_array_equal(group.previous_state.iloc[1:], group.state.iloc[:-1])
audit("Actual holdings carry continuously across annual WFO boundaries")
np.testing.assert_allclose(all_equities[:, zero_cost_id], np.array([first_kernel_result[1][:, 10]]) if N_PATHS == 1 else np.stack([daily_results.loc[daily_results.path == i, 'gross_equity'].to_numpy() for i in range(N_PATHS)]), rtol=1e-9)
audit("Zero-cost scenarios equal gross curves under the same decisions")
assert_daily_accounting(daily_results, "All paths")
audit("Wealth-normalized daily and annual equity/bond/cost contributions reconcile to net returns")

f0, rules0, over0, intra0 = first_inputs
for year in range(FUTURE_YEARS):
    row = folds.loc[(folds.path == 0) & (folds.simulation_year == year + 1)].iloc[0]
    scores = first_kernel_result[3][year]
    if np.isfinite(scores).any():
        assert int(row.config_id) == int(np.nanargmax(scores))
    else: assert row.config_id == -1
all_maxima = training_results.groupby(["path", "simulation_year"]).train_net_sharpe.max()
np.testing.assert_allclose(folds.set_index(["path", "simulation_year"]).train_net_sharpe,
                           all_maxima.reindex(folds.set_index(["path", "simulation_year"]).index),
                           rtol=1e-10, equal_nan=True)
audit("Selected configuration maximizes eligible net training Sharpe on every path/year")
for (path, year), block in daily_results.groupby(["path", "simulation_year"]):
    assert block.config_id.nunique() == 1
    expected = folds.loc[(folds.path == path) & (folds.simulation_year == year), "config_id"].iloc[0]
    assert block.config_id.iloc[0] == expected
audit("Parameters remain frozen throughout every simulated investment year")

cut = n_history + SESSIONS_PER_YEAR
prefix_features = build_features(first_frame.iloc[:cut], parameters)
pd.testing.assert_frame_equal(prefix_features, f0.iloc[:cut], rtol=1e-12, atol=1e-12)
audit("Features match when future simulated observations are removed")
if FUTURE_YEARS > 1:
    changed = first_frame.copy()
    for field in ["open", "close", "adj_close"]:
        changed.loc[cut:, f"{SIGNAL_PAIR[0]}_{field}"] *= np.linspace(1.4, 2, len(changed) - cut)
    _, changed_rules, changed_over, changed_intra = engine_inputs(changed)
    changed_result = run_simulated_wfo(changed_over, changed_intra, changed_rules, n_history, train_sessions,
                                      FUTURE_YEARS, SESSIONS_PER_YEAR, INITIAL_CAPITAL,
                                      COMMISSION_BPS / 10000, SLIPPAGE_BPS / 10000,
                                      MIN_TRAIN_ENTRIES, cost_values, base_cost_id)
    np.testing.assert_allclose(changed_result[0][:, :SESSIONS_PER_YEAR], first_kernel_result[0][:, :SESSIONS_PER_YEAR], rtol=1e-11)
    np.testing.assert_allclose(changed_result[3][:min(2, FUTURE_YEARS)], first_kernel_result[3][:min(2, FUTURE_YEARS)], rtol=1e-11, equal_nan=True)
    audit("Changing future simulated prices leaves earlier training scores and investment results unchanged")
else:
    audit_rows.append({"check": "Future perturbation requires a second simulated year", "status": "NOT APPLICABLE"})

toy = np.zeros((4, 5, 1), dtype=np.bool_)
toy[0, 1, 0] = True
toy[2, 2, 0] = True
assert advance_state(0, toy, 1, 0) == 0 and advance_state(0, toy, 2, 0) == 1
assert advance_state(1, toy, 2, 0) == 1 and advance_state(1, toy, 3, 0) == 0
audit("Synthetic signal tests enforce next-session entry/exit execution")
for old_state, new_state, expected_legs in [(0, 1, 1), (1, 0, 1), (1, 2, 2), (1, 1, 0)]:
    factor, c, s, legs, _ = cost_factor(old_state, new_state, COMMISSION_BPS / 10000, SLIPPAGE_BPS / 10000)
    assert legs == expected_legs
    np.testing.assert_allclose(factor + c + s, 1, atol=1e-15)
audit("Cost factors reconcile and switches pay two legs")
for config in sorted(set([0, int(first_kernel_result[2][0, 0])])):
    if config < 0: continue
    state = 0
    wealth = 1.0
    reference_returns = []
    entries = 0
    for day in range(n_history - train_sessions, n_history):
        old = state
        signal = day - 1
        if (state == 1 and rules0[2, signal, config]) or (state == 2 and rules0[3, signal, config]):
            state = 0
        if state == 0:
            if rules0[0, signal, config]: state = 1
            elif rules0[1, signal, config]: state = 2
        before = wealth
        wealth *= 1 + over0[day, old]
        if old != state:
            if old: wealth *= (1 - SLIPPAGE_BPS / 10000) * (1 - COMMISSION_BPS / 10000)
            if state:
                wealth /= (1 + SLIPPAGE_BPS / 10000) * (1 + COMMISSION_BPS / 10000)
                entries += 1
        wealth *= 1 + intra0[day, state]
        reference_returns.append(wealth / before - 1)
    reference_returns = np.array(reference_returns)
    reference_sharpe = reference_returns.mean() / reference_returns.std(ddof=1) * np.sqrt(252) if reference_returns.std(ddof=1) > 1e-14 and entries >= MIN_TRAIN_ENTRIES else np.nan
    np.testing.assert_allclose(reference_sharpe, first_kernel_result[3][0, config], rtol=1e-9, equal_nan=True)
audit("Compiled training scores match an independent scalar transaction-cost calculation")
np.testing.assert_allclose(trades.commission.sum(), daily_results.commission.sum(), rtol=1e-10)
np.testing.assert_allclose(trades.slippage.sum(), daily_results.slippage.sum(), rtol=1e-10)
assert len(trades) == daily_results.transaction_legs.sum()
assert trades.empty or (trades.signal_simulation_session < trades.simulation_session).all()
audit("Per-leg transaction ledger reconciles to all paid costs and delayed fills")
for i in range(len(cost_names)):
    for j in range(len(cost_names)):
        if (cost_values[i] <= cost_values[j]).all():
            assert (all_equities[:, i] >= all_equities[:, j] - 1e-6).all()
audit("Higher component-wise costs cannot improve equity under identical holding decisions")
audit("Monthly compounded returns reconcile to each simulated annual return")
roundoff_case = pd.DataFrame({
    "starting_equity": [1e14], "net_equity": [1e14], "net_return": [0.0],
    "equity_pnl": [-0.000732421875], "bond_pnl": [0.0],
    "commission": [0.0], "slippage": [0.0],
})
assert_daily_accounting(roundoff_case, "Large-equity roundoff regression")
audit("Large-equity roundoff is assessed in portfolio-relative return units")
broken_case = pd.DataFrame({
    "starting_equity": [100000.0], "net_equity": [99900.0], "net_return": [-0.001],
    "equity_pnl": [0.0], "bond_pnl": [0.0],
    "commission": [0.0], "slippage": [0.0],
})
rejected = False
try:
    assert_daily_accounting(broken_case, "Deliberately omitted transaction cost")
except AssertionError:
    rejected = True
assert rejected
audit("Accounting checks still reject a genuine missing-cost error")
audit_table = pd.DataFrame(audit_rows)
display(audit_table)
print(f"{audit_table.status.eq('PASS').sum()} audit checks passed")


## Per-simulation Sharpe tests: H0 ≤ 0.5 versus H1 > 0.5

Each path's **own simulated OOS net daily returns** is tested separately, using a studentized stationary bootstrap and influence-function Newey–West HAC standard errors. All results and block-length sensitivities are retained. The paths are never pooled into a longer historical record.

These are tests conditional on the simulated generator and fixed research design. A fraction of significant simulated paths is a robustness diagnostic, not a real-world success probability. Historical source data are reused, and both the generator and strategy design were informed by history. The earlier historical WFO test is a separate analysis.

The bootstrap pivot is `(bootstrap Sharpe − path's observed Sharpe) / bootstrap standard error`, compared with `(observed Sharpe − 0.5) / observed standard error`. This supports the nonzero null without incorrectly centering returns at zero. See [Ledoit and Wolf](https://www.ledoit.net/Robust_Sharpe_2008.pdf).

In [ ]:
from numba import njit

@njit
def sharpe_hac_stat(values, lags):
    n = len(values)
    mean = values.mean()
    variance = 0.0
    for value in values:
        variance += (value - mean) ** 2
    variance /= n
    if variance <= 1e-28:
        return np.nan, np.nan
    sigma = np.sqrt(variance)
    correction = np.sqrt((n - 1.0) / n)
    scale = np.sqrt(252.0) * correction
    sharpe = scale * mean / sigma
    influence = np.empty(n)
    for i in range(n):
        centered = values[i] - mean
        influence[i] = scale * (centered / sigma - mean / (2 * sigma ** 3) * (centered ** 2 - variance))
    influence -= influence.mean()
    long_run_variance = np.dot(influence, influence) / n
    bandwidth = min(lags, n - 1)
    for lag in range(1, bandwidth + 1):
        weight = 1 - lag / (bandwidth + 1.0)
        covariance = np.dot(influence[lag:], influence[:-lag]) / n
        long_run_variance += 2 * weight * covariance
    standard_error = np.sqrt(max(long_run_variance, 0.0) / n)
    return sharpe, standard_error

@njit
def studentized_sharpe_bootstrap(values, replicates, block_length, lags, seed):
    np.random.seed(seed)
    n = len(values)
    observed, observed_se = sharpe_hac_stat(values, lags)
    output = np.empty((replicates, 3))
    sampled = np.empty(n)
    accepted = 0
    attempts = 0
    while accepted < replicates and attempts < replicates * 20:
        attempts += 1
        index = 0
        for i in range(n):
            if i == 0 or np.random.random() < 1.0 / block_length:
                index = np.random.randint(0, n)
            else:
                index = (index + 1) % n
            sampled[i] = values[index]
        estimate, se = sharpe_hac_stat(sampled, lags)
        if np.isfinite(estimate) and np.isfinite(se) and se > 1e-14:
            output[accepted, 0] = estimate
            output[accepted, 1] = se
            output[accepted, 2] = (estimate - observed) / se
            accepted += 1
    return output[:accepted], attempts


In [ ]:
if not (0 < SHARPE_ALPHA < 1 and SHARPE_BOOTSTRAPS >= 1000 and SHARPE_HAC_LAGS >= 0):
    raise ValueError("Invalid Sharpe-test settings")
if SHARPE_PRIMARY_BLOCK not in SHARPE_BLOCK_LENGTHS or min(SHARPE_BLOCK_LENGTHS) < 1:
    raise ValueError("Invalid primary block or sensitivity lengths")
if len(set(SHARPE_BLOCK_LENGTHS)) != len(SHARPE_BLOCK_LENGTHS):
    raise ValueError("Sharpe block lengths must be unique")
seed_count = N_PATHS * len(SHARPE_BLOCK_LENGTHS)
test_seeds = [int(seed.generate_state(1)[0]) for seed in np.random.SeedSequence(SHARPE_TEST_SEED).spawn(seed_count)]
per_path_sharpe_rows = []
saved_sharpe_draws = []
first_path_primary_draws = None
first_path_primary_seed = None
for path_id in range(N_PATHS):
    returns = np.ascontiguousarray(daily_results.loc[daily_results.path == path_id, "net_return"].to_numpy(dtype=float))
    if len(returns) != future_sessions or not np.isfinite(returns).all():
        raise ValueError("Each path must have exactly its own finite simulated OOS return series")
    observed, se = sharpe_hac_stat(returns, SHARPE_HAC_LAGS)
    np.testing.assert_allclose(observed, metrics.loc[path_id, "Sharpe"], rtol=1e-10)
    if not np.isfinite(se) or se <= 1e-14:
        raise ValueError(f"Path {path_id} has undefined Sharpe inference")
    observed_t = (observed - SHARPE_NULL) / se
    for k, block in enumerate(SHARPE_BLOCK_LENGTHS):
        seed = test_seeds[path_id * len(SHARPE_BLOCK_LENGTHS) + k]
        draws, attempts = studentized_sharpe_bootstrap(returns, SHARPE_BOOTSTRAPS, block, SHARPE_HAC_LAGS, seed)
        if len(draws) != SHARPE_BOOTSTRAPS:
            raise ValueError("Too many degenerate bootstrap draws")
        pivots = draws[:, 2]
        p_value = (1 + np.count_nonzero(pivots >= observed_t)) / (len(pivots) + 1)
        critical = float(np.quantile(pivots, 1 - SHARPE_ALPHA, method="higher"))
        lower_bound = observed - critical * se
        per_path_sharpe_rows.append({
            "path": path_id, "block_length": block, "primary": block == SHARPE_PRIMARY_BLOCK,
            "simulated_observations": len(returns), "observed_sharpe": observed,
            "null_sharpe": SHARPE_NULL, "HAC_standard_error": se, "observed_t": observed_t,
            "one_sided_p_value": p_value, "one_sided_lower_bound": lower_bound,
            "confidence_level": 1 - SHARPE_ALPHA, "reject_null": p_value < SHARPE_ALPHA,
            "replicates": len(draws), "discarded_degenerate_draws": attempts - len(draws), "seed": seed,
        })
        if SAVE_SHARPE_DRAWS:
            table = pd.DataFrame(draws, columns=["bootstrap_sharpe", "bootstrap_HAC_se", "studentized_pivot"])
            table["path"] = path_id
            table["block_length"] = block
            table["replicate"] = np.arange(len(draws))
            saved_sharpe_draws.append(table)
        if path_id == 0 and block == SHARPE_PRIMARY_BLOCK:
            first_path_primary_draws = draws.copy()
            first_path_primary_seed = seed
    primary_result = per_path_sharpe_rows[-len(SHARPE_BLOCK_LENGTHS):][SHARPE_BLOCK_LENGTHS.index(SHARPE_PRIMARY_BLOCK)]
    print(f"Sharpe test path {path_id + 1}/{N_PATHS}: S={observed:.3f}, p={primary_result['one_sided_p_value']:.4f}", flush=True)
per_path_sharpe_test = pd.DataFrame(per_path_sharpe_rows)
primary_path_tests = per_path_sharpe_test.loc[per_path_sharpe_test.primary].set_index("path")
sharpe_test_summary = per_path_sharpe_test.groupby("block_length").agg(
    paths=("path", "size"), rejection_fraction=("reject_null", "mean"),
    median_p_value=("one_sided_p_value", "median"),
    median_lower_bound=("one_sided_lower_bound", "median"),
)
display(primary_path_tests[["observed_sharpe", "null_sharpe", "one_sided_p_value", "one_sided_lower_bound", "reject_null"]].style.format({
    "observed_sharpe": "{:.3f}", "null_sharpe": "{:.2f}", "one_sided_p_value": "{:.4f}", "one_sided_lower_bound": "{:.3f}",
}))
display(sharpe_test_summary)
print(f"Primary block {SHARPE_PRIMARY_BLOCK}: {primary_path_tests.reject_null.sum()} of {N_PATHS} simulated paths reject H0 at {SHARPE_ALPHA:.0%}.")
print("This is conditional simulation evidence. Rejection frequency is not the probability of a real-world edge; no path is selected or discarded.")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), constrained_layout=True)
path_numbers = primary_path_tests.index.to_numpy()
axes[0].scatter(path_numbers, primary_path_tests.observed_sharpe, label="Observed simulated-path Sharpe", color="tab:blue")
axes[0].scatter(path_numbers, primary_path_tests.one_sided_lower_bound, label=f"One-sided {1 - SHARPE_ALPHA:.0%} lower bound", color="tab:orange", marker="_")
for path_id, row in primary_path_tests.iterrows():
    axes[0].plot([path_id, path_id], [row.one_sided_lower_bound, row.observed_sharpe], color="gray", alpha=.4)
axes[0].axhline(SHARPE_NULL, color="red", linestyle="--", label=f"Null boundary {SHARPE_NULL}")
axes[0].set_xlabel("Simulated path (all retained)")
axes[0].set_ylabel("Reported annualized Sharpe")
axes[0].set_title(f"Per-path Sharpe inference: primary block {SHARPE_PRIMARY_BLOCK}, not pooled")
axes[0].legend(fontsize=8)
image = axes[1].imshow(per_path_sharpe_test.pivot(index="path", columns="block_length", values="one_sided_p_value").to_numpy(),
                        aspect="auto", cmap="viridis_r", vmin=0, vmax=1)
ordered_blocks = sorted(SHARPE_BLOCK_LENGTHS)
axes[1].set_xticks(range(len(ordered_blocks)), ordered_blocks)
axes[1].set_yticks(range(N_PATHS), [f"Path {i}" for i in range(N_PATHS)])
axes[1].set_xlabel("Mean bootstrap block length (sessions)")
axes[1].set_title("One-sided p-values by simulated path and block length")
fig.colorbar(image, ax=axes[1]).set_label("p-value")
fig.suptitle(f"SIMULATION-CONDITIONAL TESTS: H0 Sharpe <= {SHARPE_NULL}; H1 > {SHARPE_NULL}")
fig.savefig(output_dir / "simulated_path_sharpe_tests.png", dpi=140)
plt.show()


## Per-path inference audit

In [ ]:
sharpe_sim_checks = []
assert len(primary_path_tests) == N_PATHS
assert primary_path_tests.simulated_observations.eq(future_sessions).all()
sharpe_sim_checks.append({"check": "Each path tested separately using only its own simulated OOS returns", "status": "PASS"})
np.testing.assert_allclose(primary_path_tests.observed_sharpe, metrics.Sharpe, rtol=1e-10)
sharpe_sim_checks.append({"check": "Tested Sharpe matches each path's net performance statistic", "status": "PASS"})
x = np.ascontiguousarray(daily_results.loc[daily_results.path == 0, "net_return"].to_numpy())
replay, _ = studentized_sharpe_bootstrap(x, 64, SHARPE_PRIMARY_BLOCK, SHARPE_HAC_LAGS, first_path_primary_seed)
np.testing.assert_allclose(replay, first_path_primary_draws[:64], rtol=1e-12)
sharpe_sim_checks.append({"check": "Inference bootstrap is reproducible from the recorded seed", "status": "PASS"})
observed0, se0 = sharpe_hac_stat(x, SHARPE_HAC_LAGS)
scaled0, scaled_se0 = sharpe_hac_stat(x * 10, SHARPE_HAC_LAGS)
np.testing.assert_allclose([observed0, se0], [scaled0, scaled_se0], rtol=1e-10)
sharpe_sim_checks.append({"check": "Sharpe inference is invariant to positive return rescaling", "status": "PASS"})
for _, row in per_path_sharpe_test.iterrows():
    if abs(row.one_sided_p_value - SHARPE_ALPHA) > 2 / (row.replicates + 1):
        assert bool(row.reject_null) == bool(row.one_sided_lower_bound > SHARPE_NULL)
sharpe_sim_checks.append({"check": "P-values agree with one-sided confidence-bound decisions outside Monte Carlo boundary", "status": "PASS"})
assert len(metrics) == N_PATHS and len(folds) == N_PATHS * FUTURE_YEARS
sharpe_sim_checks.append({"check": "No simulated paths were filtered or promoted based on test significance", "status": "PASS"})
sharpe_sim_audit = pd.DataFrame(sharpe_sim_checks)
display(sharpe_sim_audit)


## Export all paths, selections, costs and provenance; no performance-based path filtering

In [ ]:
per_path_sharpe_test.to_csv(output_dir / "per_path_sharpe_tests.csv", index=False)
sharpe_test_summary.to_csv(output_dir / "sharpe_test_sensitivity.csv")
sharpe_sim_audit.to_csv(output_dir / "sharpe_simulation_audit.csv", index=False)
if SAVE_SHARPE_DRAWS:
    pd.concat(saved_sharpe_draws, ignore_index=True).to_csv(output_dir / "per_path_sharpe_bootstrap_draws.csv.gz", index=False, compression="gzip")
accounting_diagnostics.to_csv(output_dir / "accounting_residuals.csv")
import gzip
raw_input = source_path.read_bytes()
snapshot_payload = gzip.decompress(raw_input) if raw_input.startswith(bytes([31, 139])) else raw_input
snapshot = output_dir / "historical_input_snapshot.csv.gz"
with gzip.open(snapshot, "wb") as handle:
    handle.write(snapshot_payload)
assert hashlib.sha256(gzip.decompress(snapshot.read_bytes())).hexdigest() == hashlib.sha256(snapshot_payload).hexdigest()
pool_table = pd.DataFrame({"source_row": pool_rows, "source_date": prices.index[pool_rows]})
for j, ticker in enumerate(TICKERS):
    pool_table[f"{ticker}_overnight_return"] = source_overnight[pool_rows, j]
    pool_table[f"{ticker}_intraday_return"] = source_intraday[pool_rows, j]
pool_table.to_csv(output_dir / "frozen_calibration_return_pool.csv", index=False)
trades.to_csv(output_dir / "simulation_trades.csv.gz", index=False, compression="gzip")
monthly_table.to_csv(output_dir / "monthly_yearly_returns.csv")
selection_frequency.to_csv(output_dir / "parameter_selection_frequency.csv", index=False)
metrics.to_csv(output_dir / "path_metrics.csv")
summary.to_csv(output_dir / "simulation_percentiles.csv")
folds.to_csv(output_dir / "wfo_folds.csv", index=False)
training_results.to_csv(output_dir / "training_grid_scores.csv.gz", index=False, compression="gzip")
daily_results.to_csv(output_dir / "simulation_daily.csv.gz", index=False, compression="gzip")
source_trace.to_csv(output_dir / "sampled_source_blocks.csv.gz", index=False, compression="gzip")
annual_returns.to_csv(output_dir / "annual_returns.csv")
attribution.to_csv(output_dir / "annual_return_contributions.csv")
audit_table.to_csv(output_dir / "simulation_audit.csv", index=False)
if SAVE_SYNTHETIC_PRICES:
    pd.concat(all_synthetic_prices, ignore_index=True).to_csv(output_dir / "synthetic_future_prices.csv.gz", index=False, compression="gzip")
cost_output = []
for path_id in range(N_PATHS):
    table = pd.DataFrame(all_equities[path_id].T, columns=cost_names)
    table.insert(0, "simulation_session", np.arange(1, future_sessions + 1))
    table.insert(0, "path", path_id)
    cost_output.append(table)
pd.concat(cost_output, ignore_index=True).to_csv(output_dir / "cost_path_equities.csv.gz", index=False, compression="gzip")
manifest = {
    "kind": "Conditional stationary-block bootstrap simulation; not historical OOS evidence",
    "pair": list(PAIR), "signal_pair": list(SIGNAL_PAIR), "rule_set": RULE_SET,
    "origin": origin.date().isoformat(), "calibration_start": prices.index[pool_rows[0]].date().isoformat(),
    "calibration_years": CALIBRATION_YEARS, "calibration_scope": "entire joint dataset" if CALIBRATION_YEARS is None else "trailing window", "mean_block_length": MEAN_BLOCK_LENGTH,
    "replacement": True, "shared_source_indices_across_assets": True,
    "master_seed": MASTER_SEED, "child_seeds": seeds, "paths": N_PATHS, "future_years": FUTURE_YEARS,
    "sessions_per_year": SESSIONS_PER_YEAR, "training_sessions": train_sessions,
    "investment_sessions": SESSIONS_PER_YEAR, "grid": GRIDS[RULE_SET],
    "selection": "Maximum preceding-window net Sharpe, zero risk-free rate; no path filtering",
    "commission_bps": COMMISSION_BPS, "slippage_bps": SLIPPAGE_BPS,
    "cost_paths_bps": COST_PATHS_BPS, "cost_paths_reoptimized": False,
    "sharpe_test": {
        "null": SHARPE_NULL, "alpha": SHARPE_ALPHA, "primary_block": SHARPE_PRIMARY_BLOCK,
        "block_lengths": SHARPE_BLOCK_LENGTHS, "HAC_lags": SHARPE_HAC_LAGS,
        "replicates_per_path_per_block": SHARPE_BOOTSTRAPS, "seed": SHARPE_TEST_SEED,
        "scope": "Each simulated OOS path separately; conditional on generator; no pooling; no path filtering",
        "primary_rejections": int(primary_path_tests.reject_null.sum()), "paths_tested": N_PATHS,
    },
    "initial_capital": INITIAL_CAPITAL, "audits_passed": int(audit_table.status.eq("PASS").sum()),
    "historical_input_snapshot": snapshot.name,
    "source_csv": str(source_path), "source_sha256": hashlib.sha256(snapshot_payload).hexdigest(),
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
(output_dir / "run_config.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(f"All simulated results saved separately to {output_dir.resolve()}")
